In [ ]:
# Transformando a lista em DataFrame
nomes_ativos = [
    'ITSA4', 'ABEV3', 'B3SA3', 'BBAS3', 'BBDC4', 
    'ELET3', 'ITUB4', 'PETR4', 'VALE3', 'WEGE3'
]

df_log_completo = pd.concat(lista_log, axis=1)
df_log_completo.columns = nomes_ativos

df_log_completo
#Transformando o Dataframe em csv

df_log_completo.to_excel('dados_para_visualizar.xlsx', index=True)

In [2]:
import yfinance as yf
import pandas as pd

In [ ]:
# Jeito mais elegante de puxar os tickers
# Tickers principais do Ibovespa (top recorrentes)
tickers = [
    "VALE3.SA",
    "PETR4.SA",
    "PETR3.SA",
    "ITUB4.SA",
    "BBDC4.SA",
    "BBAS3.SA",
    "ABEV3.SA"
]

# Período
start_date = "2021-01-01"
end_date = "2025-12-31"

# Baixar dados
data = yf.download(tickers, start=start_date, end=end_date)["Close"]

# Limpar dados
data = data.dropna(how="all")

# Renomear colunas (tirar .SA)
data.columns = [col.replace(".SA", "") for col in data.columns]

# Salvar CSV
data.to_csv("ibov_top_acoes_close.csv")

print(data.head())

In [ ]:
# Calculando a série do spread usando os pesos do teste
df_consolidado['Spread_BBDC_VALE'] = df_consolidado['BBDC4.SA'] - 1.7893 * df_consolidado['VALE3.SA']

# Plotando o gráfico do Spread
plt.figure(figsize=(12, 5))
plt.plot(df_consolidado['Spread_BBDC_VALE'], color='purple', label='Spread (BBDC4 - 1.7893*VALE3)')
plt.axhline(df_consolidado['Spread_BBDC_VALE'].mean(), color='red', linestyle='--', label='Média do Spread')
plt.title('Série Temporal do Spread Cointegrado (Reversão à Média)')
plt.xlabel('Data')
plt.ylabel('Valor do Spread')
plt.legend()
plt.grid(True)
plt.show()

In [5]:
teste = yf.download("AXIA3.SA", start="2021-01-01", end="2025-12-31", auto_adjust=True)["Close"]

teste.index = teste.index.date

(teste.pct_change().fillna(0) + 1).cumprod() - 1

[*********************100%***********************]  1 of 1 completed


Ticker,AXIA3.SA
2021-01-04,0.000000
2021-01-05,-0.025821
2021-01-06,-0.041538
2021-01-07,-0.048274
2021-01-08,-0.018243
...,...
2025-12-22,0.712973
2025-12-23,0.758746
2025-12-26,0.756281
2025-12-29,0.756985


In [ ]:
# 1. Isolando a cesta definitiva com os 8 ativos I(1) legítimos
# (Garante a remoção caso os nomes tenham ou não o sufixo '.SA')
colunas_remover = [col for col in ['GGBR4.SA', 'B3SA3.SA', 'GGBR4', 'B3SA3'] if col in df_consolidado.columns]
cesta_definitiva = df_consolidado.drop(columns=colunas_remover)

# 2. Executando o Teste de Johansen
# det_order = 0 -> Inclui uma constante no vetor de cointegração (padrão para preços)
# k_ar_diff = 1 -> Número de lags na primeira diferença (corresponde a um VAR de ordem 2 em nível)
resultado_johansen = coint_johansen(cesta_definitiva, det_order=0, k_ar_diff=1)

# 3. Estruturando a Tabela do Teste do Traço (Trace Statistic)
df_traco = pd.DataFrame({
    'Hipótese Nula (H0)': [f'r <= {i}' for i in range(len(cesta_definitiva.columns))],
    'Estatística do Traço': resultado_johansen.lr1,
    'V.C. 90%': resultado_johansen.cvt[:, 0],
    'V.C. 95%': resultado_johansen.cvt[:, 1],
    'V.C. 99%': resultado_johansen.cvt[:, 2]
})
# Identifica se a estatística rejeita a hipótese nula a 95% de confiança
df_traco['Rejeita H0 (95%)?'] = df_traco['Estatística do Traço'] > df_traco['V.C. 95%']

# 4. Estruturando a Tabela do Teste do Máximo Autovalor (Max Eigenvalue)
df_max_eigen = pd.DataFrame({
    'Hipótese Nula (H0)': [f'r == {i}' for i in range(len(cesta_definitiva.columns))],
    'Estatística Max Eigen': resultado_johansen.lr2,
    'V.C. 90%': resultado_johansen.cvm[:, 0],
    'V.C. 95%': resultado_johansen.cvm[:, 1],
    'V.C. 99%': resultado_johansen.cvm[:, 2]
})
df_max_eigen['Rejeita H0 (95%)?'] = df_max_eigen['Estatística Max Eigen'] > df_max_eigen['V.C. 95%']

# 5. EXTRAÇÃO DO VETOR PRINCIPAL: Os pesos do primeiro portfólio estável
# Os vetores de cointegração estão nas colunas da matriz .evec
vetor_principal = resultado_johansen.evec[:, 0]
# Normalizando os pesos dividindo pelo peso do primeiro ativo (facilita a montagem do spread)
vetor_normalizado = vetor_principal / vetor_principal[0]

df_pesos_spread = pd.DataFrame({
    'Ativo': cesta_definitiva.columns,
    'Peso no Spread (Beta)': np.round(vetor_normalizado, 4)
})

# --- EXIBIÇÃO DOS RESULTADOS ---
print("="*30 + " 1. TESTE DO TRAÇO " + "="*30)
print(df_traco.to_string(index=False))

print("\n" + "="*25 + " 2. TESTE DO MÁXIMO AUTOVALOR " + "="*25)
print(df_max_eigen.to_string(index=False))

print("\n" + "="*20 + " 3. VETOR DE COINTEGRAÇÃO PRINCIPAL " + "="*20)
print(df_pesos_spread.to_string(index=False))